# Table Structure Recognition Dataset Profiler

This notebook profiles and analyzes structural properties of two table recognition datasets: **SciTSR** (scientific papers) and **A25** (multiple domains).

In [ ]:

from pathlib import Path
import json, re
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from PIL import Image

SCITSR_ROOT   = Path("data/SciTSR/test")                     
SCITSR_SPLITS = ["train", "test"]
A25_ROOT    = Path("data/A25/input_images")   
A25_DOMAINS = ["Biology", "CompSci", "ICDAR", "MatSci"]

SCITSR_VISION_PRED = Path("data/outputs/vision_expert_gemma4_SciTSR_run")   
SCITSR_TEXT_PRED   = Path("data/outputs/text_expert_gemma4_SciTSR_run")     
A25_VISION_PRED  = Path("data/outputs/vision_expert_gemma4_run")     
A25_TEXT_PRED    = Path("data/outputs/text_expert_gemma4_run")        

ONLY_EVALUATED = True   # True = count only tables BOTH agents predicted 
IMG_EXTS = (".png")

# ---------------------------------------------------------------------
def table_metrics(cells):
    if not cells:
        return None
    
    n_rows = max(c["er"] for c in cells) + 1
    n_cols = max(c["ec"] for c in cells) + 1

    n_multirow = sum(1 for c in cells if c["er"] > c["sr"])
    n_multicol = sum(1 for c in cells if c["ec"] > c["sc"])

    covered = sum((c["er"]-c["sr"]+1)*(c["ec"]-c["sc"]+1) for c in cells)
    density = covered / (n_rows*n_cols) if n_rows*n_cols else 0.0

    return dict(
        n_rows=n_rows, 
        n_cols=n_cols, 
        n_cells=len(cells),
        n_multirow=n_multirow, 
        n_multicol=n_multicol,
        has_multirow=n_multirow > 0, 
        has_multicol=n_multicol > 0,
        density=density
    )

def find_image(stem, *dirs):

    for d in dirs:

        if d is None or not Path(d).exists():
            continue

        for ext in IMG_EXTS:
            p = Path(d) / f"{stem}{ext}"

            if p.exists():
                return p
            
    return None

def stems_in(d, pattern="*.json"):

    return {p.stem for p in Path(d).glob(pattern)} if Path(d).exists() else set()

def parse_scitsr_json(path):

    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    cells = data.get("cells", data) if isinstance(data, dict) else data

    out = []
    for c in cells:

        sr = c.get("start_row", c.get("sr"))
        sc = c.get("start_col", c.get("sc"))

        if sr is None or sc is None:
            continue

        er = c.get("end_row", c.get("er", sr))
        ec = c.get("end_col", c.get("ec", sc))

        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec)
        })

    return out

def parse_a25_xml(path):

    try:
        root = ET.parse(path).getroot()
    except Exception:
        return []
    
    out = []
    for c in root.iter("cell"):

        sr = c.get("start_row")
        sc = c.get("start_col")

        if sr is None or sc is None:
            continue

        er = c.get("end_row", sr) 
        ec = c.get("end_col", sc)

        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec)
        })

    return out

def profile_scitsr():
    rows = []

    for split in SCITSR_SPLITS:
        
        gt_dir  = SCITSR_ROOT / split / "structure_processed"
        img_dir = SCITSR_ROOT / split / "img"

        if not gt_dir.exists():
            continue

        used = None

        if ONLY_EVALUATED:

            vis = stems_in(SCITSR_VISION_PRED / split / "predictions")
            txt = stems_in(SCITSR_TEXT_PRED / split / "predictions")

            used = vis & txt
            print(f"[scitsr/{split}] GT={len(list(gt_dir.glob('*.json')))} "
                  f"vision_pred={len(vis)} text_pred={len(txt)} -> evaluated={len(used)}")
            
        for gt in sorted(gt_dir.glob("*.json")):

            if used is not None and gt.stem not in used:
                continue

            m = table_metrics(parse_scitsr_json(gt))

            if m is None:
                continue

            rows.append({
                "dataset": "SciTSR", 
                "domain": "Scientific (papers)", 
                "split": split,
                "table_id": f"{split}::{gt.stem}", **m,
            })
    return pd.DataFrame(rows)

def profile_a25():
    rows = []

    for domain in A25_DOMAINS:

        gt_dir = A25_ROOT / domain / "xmls"
        if not gt_dir.exists():
            continue

        img_dirs = [A25_ROOT / domain / "images", A25_ROOT / domain / "img", A25_ROOT / domain]

        used = None

        if ONLY_EVALUATED:

            vis = stems_in(A25_VISION_PRED / domain / "predictions")
            txt = stems_in(A25_TEXT_PRED / domain / "nougat" / "predictions")
            used = vis & txt

            print(f"[A25/{domain}] GT={len(list(gt_dir.glob('*.xml')))} "
                  f"vision_pred={len(vis)} text_pred={len(txt)} -> evaluated={len(used)}")
            
        for gt in sorted(gt_dir.glob("*.xml")):

            if used is not None and gt.stem not in used:
                continue

            m = table_metrics(parse_a25_xml(gt))

            if m is None:
                continue

            rows.append({
                "dataset": "a25", 
                "domain": domain, 
                "split": "all",
                "table_id": f"{domain}::{gt.stem}", **m,
            })
    return pd.DataFrame(rows)

scitsr_tables = profile_scitsr()
a25_tables  = profile_a25()
per_table = pd.concat([scitsr_tables, a25_tables], ignore_index=True)

print(f"\nProfiled {len(per_table)} evaluated tables "
      f"(SciTSR={len(scitsr_tables)}, A25={len(a25_tables)})\n")


def summarize(df, label, domain_label):
    s = pd.Series(dtype=object)
    s["Dataset"] = label
    s["Domain"] = domain_label
    s["# Tables"] = len(df)
    s["# cells (total)"] = int(df["n_cells"].sum())
    s["Rows (mean)"] = round(df["n_rows"].mean(), 1)
    s["Rows (max)"] = int(df["n_rows"].max())
    s["Cols (mean)"] = round(df["n_cols"].mean(), 1)
    s["Cols (max)"] = int(df["n_cols"].max())
    s["% multirow"] = round(100 * df["has_multirow"].mean(), 1)
    s["% multicol"] = round(100 * df["has_multicol"].mean(), 1)
    s["Density (mean)"] = round(df["density"].mean(), 2)
    return s

summary_rows = []
if len(scitsr_tables):
    summary_rows.append(summarize(scitsr_tables, "SciTSR", "Scientific (papers)"))

if len(a25_tables):
    dom = ", ".join(sorted(a25_tables["domain"].unique()))
    summary_rows.append(summarize(a25_tables, "a25", dom))

summary = pd.DataFrame(summary_rows)

a25_by_domain = pd.DataFrame(
    [summarize(g, f"a25 / {d}", d) for d, g in a25_tables.groupby("domain")]
) if len(a25_tables) else pd.DataFrame()

print("================ DATASET SUMMARY ================")
print(summary.to_string(index=False))

if len(a25_by_domain):
    print("\n--------- a25 per-domain breakdown ----------")
    print(a25_by_domain.to_string(index=False))